# Aggregated multi-setup analysis display

Here we display the results computed in the notebook *CMIP_analysis_agregate*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# --- Loading the aggregated pkl file (built in CMIP_analysis_agregate.ipynb) ---
pkl_dir = "/glade/u/home/tsalin/CMIP/model_evaluation/agregate_display"
df = pd.read_pickle(f"{pkl_dir}/agregate_all_setups_df.pkl")

# r2_by_climate is a dict by scenario -> we extract ssp585 / historical
df["r2_ssp585"] = df["r2_by_climate"].apply(lambda d: d["ssp585"])
df["r2_historical"] = df["r2_by_climate"].apply(lambda d: d["historical"])

df_suite = df.copy()

print(f"Number of rows (all setups, all hyperparameters) : {len(df)}")

## Filters (hyperparameters & setups)

In [ ]:
# --- Hyperparameter filters ---
# For each hyperparameter column, choose one of:
#   "all"          -> keep every value for that column (NaN included)
#   value          -> keep only rows equal to this value (NaN excluded)
#   (value, True)  -> keep rows equal to this value, plus rows where it's NaN
hyperparam_filters = {
    "num_sample": "all",
    "chosen_autoencoder_type": "all",
    "inv_alignment_method": "all",
    "variable": "all",
    "val_fraction": "all",
    "test_fraction": "all",
    "cera_lambda_align": "all",
    "cera_lambda_pred": "all",
}

# --- Setup filter ---
# "all" to keep every setup, or a list of the setup names to keep, e.g.:
# selected_setups = ["CERA", "CERA SWDN", "Baseline ClimaX"]
selected_setups = ["CERA", "CERA SWDN", "Baseline ClimaX"]


def apply_hyperparam_filters(data, filters):
    mask = pd.Series(True, index=data.index)
    for col, spec in filters.items():
        if spec == "all":
            continue
        value, include_nan = spec if isinstance(spec, tuple) else (spec, False)
        col_mask = data[col] == value
        if include_nan:
            col_mask |= data[col].isna()
        mask &= col_mask
    return data[mask].reset_index(drop=True)


# Always rebuilt from df_suite (the full, unfiltered table) so re-running this cell with
# new filter values never filters an already-filtered df from a previous run.
df = apply_hyperparam_filters(df_suite, hyperparam_filters)
if selected_setups != "all":
    df = df[df["setup"].isin(selected_setups)].reset_index(drop=True)

print(f"Number of rows after filtering : {len(df)}")

In [ ]:
setups = sorted(df["setup"].unique())

color_map = {
    "CERA full latent": "#0b3d6b",
    "Baseline ClimaX": "#c9a227",
    "Baseline No Align": "#808080",
    "CERA": "#2a78d6",
    "CERA MLP SWD": "#2ad64c",
    "CERA SWDN": "#d62a8e",
    "CERA MLP SWDN": "#151806",
    "CERA adversarial" : "#d6492a",
    "CERA seasonal": "#52565a",
    "CERA seasonal SWDN": "#b92ad6",
}


colors = df["setup"].map(color_map)

In [ ]:
# --- Graph 1 : alignment_swd_hist_ssp585 (x) vs classifier_accuracy_mlp (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["alignment_swd_hist_ssp585"], sub["classifier_accuracy_mlp"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("alignment_swd_hist_ssp585")
ax.set_ylabel("classifier_accuracy_mlp")
ax.set_title("Classifier accuracy vs. alignment SWD (historical vs ssp585)")
ax.set_xlim(0, 0.05)
ax.set_ylim(0.2, 0.46)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 2 : classifier_accuracy_mlp (x) vs r2_by_climate['ssp585'] (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["classifier_accuracy_mlp"], sub["r2_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("classifier_accuracy_mlp")
ax.set_ylabel("r2_by_climate (ssp585)")
ax.set_title("R2 (ssp585) vs. classifier accuracy")
ax.set_ylim(0.2, 0.9)
ax.set_xlim(0.20, 0.5)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 3 : alignment_swd_hist_ssp585 (x) vs ratio_median (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["alignment_swd_hist_ssp585"], sub["ratio_median"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("alignment_swd_hist_ssp585")
ax.set_ylabel("ratio_median")
ax.set_title("Ratio median vs. alignment SWD (historical vs ssp585)")
ax.set_ylim(0, 40)
ax.set_xlim(0, 0.08)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 4 : alignment_swd_hist_ssp585 (x) vs r2_by_climate['ssp585'] (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["alignment_swd_hist_ssp585"], sub["r2_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("alignment_swd_hist_ssp585")
ax.set_ylabel("r2_by_climate (ssp585)")
ax.set_title("R2 (ssp585) vs. alignment SWD (historical vs ssp585)")
ax.set_ylim(0, 1)
ax.set_xlim(0, 0.1)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 5 : ratio_median (x) vs r2_by_climate['ssp585'] (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["ratio_median"], sub["r2_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("ratio_median")
ax.set_ylabel("r2_by_climate (ssp585)")
ax.set_title("R2 (ssp585) vs. ratio median")
ax.set_ylim(0, 1)
ax.set_xlim(0, 50)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 6 : ratio_median (x) vs classifier_accuracy_mlp (y) ---
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["ratio_median"], sub["classifier_accuracy_mlp"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.set_xlabel("ratio_median")
ax.set_ylabel("classifier_accuracy_mlp")
ax.set_title("classifier_accuracy_mlp vs. ratio median")
ax.set_ylim(0.19, 0.46)
ax.set_xlim(1, 4)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# Graph 7 - Delta between r2 historical and r2 ssp585
df["delta_r2_hist_ssp585"] = df["r2_by_climate"].apply(
    lambda d: d["historical"] - d["ssp585"]
)

fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["alignment_swd_hist_ssp585"], sub["delta_r2_hist_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.axhline(0, color="#898781", linewidth=1, linestyle="--")  # reference line at y=0
ax.set_xlabel("alignment_swd_hist_ssp585")
ax.set_ylabel("delta r2 (historical - ssp585)")
ax.set_title("Delta R2 (historical - ssp585) vs. alignement SWD")
ax.set_ylim(-0.4, 0.2)
ax.set_xlim(0, 0.08)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 8 : delta r2 (historical - ssp585) vs classifier_accuracy_mlp ---

fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["classifier_accuracy_mlp"], sub["delta_r2_hist_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.axhline(0, color="#898781", linewidth=1, linestyle="--")  # reference line at y=0
ax.set_xlabel("classifier_accuracy_mlp")
ax.set_ylabel("delta r2 (historical - ssp585)")
ax.set_title("Delta R2 (historical - ssp585) vs. classifier accuracy")
ax.set_ylim(-0.4, 0.5)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- Graph 9 : delta r2 (historical - ssp585) vs ratio_median ---

fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["ratio_median"], sub["delta_r2_hist_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.axhline(0, color="#898781", linewidth=1, linestyle="--")  # reference line at y=0
ax.set_xlabel("ratio_median")
ax.set_ylabel("delta r2 (historical - ssp585)")
ax.set_title("Delta R2 (historical - ssp585) vs. ratio median")
ax.set_ylim(-0.4, 0.2)
ax.set_xlim(1, 4)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for s in setups:
    sub = df[df["setup"] == s]
    ax.scatter(
        sub["r2_historical"], sub["delta_r2_hist_ssp585"],
        color=color_map[s], label=s, s=70, edgecolor="white", linewidth=0.5,
    )
ax.axhline(0, color="#898781", linewidth=1, linestyle="--")  # reference line at y=0
ax.set_xlabel("r2_historical")
ax.set_ylabel("delta r2 (historical - ssp585)")
ax.set_title("Delta R2 (historical - ssp585) vs. r2_historical")
ax.set_ylim(-0.4, 0.5)
ax.set_xlim(0, 1)
ax.legend(title="Setup", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()